# 🔍 YOLO를 활용한 실시간 객체 탐지

## 🎯 학습 목표
- **컴퓨터 비전 기초** - 이미지 읽기, 처리의 기본 개념 이해
- **YOLO(You Only Look Once)** 이해
- **실시간 객체 탐지** 시스템 구현
- **컴퓨터 비전**의 실무 활용 경험
- **개인 이미지**를 활용한 맞춤형 프로젝트

## 💡 컴퓨터 비전이란?
**컴퓨터 비전(Computer Vision)**은 컴퓨터가 이미지나 비디오를 이해하고 분석하는 인공지능 분야입니다.

### 🔬 컴퓨터 비전의 주요 분야
| 분야 | 설명 | 예시 |
|------|------|------|
| 이미지 분류 | 이미지가 어떤 종류인지 분류 | "이것은 고양이 사진입니다" |
| 객체 탐지 | 이미지에서 물체의 위치와 종류 파악 | 자율주행차의 보행자 인식 |
| 이미지 분할 | 이미지의 각 픽셀이 어떤 물체인지 구분 | 의료 영상에서 종양 영역 표시 |
| 얼굴 인식 | 얼굴을 찾고 누구인지 식별 | 스마트폰 얼굴 인식 잠금해제 |

## 💡 YOLO란?
**YOLO(You Only Look Once)**는 이미지를 보고 객체를 탐지하는 Object Detection 알고리즘입니다.

![image](https://raw.githubusercontent.com/simsimee/HKNU_lecture/refs/heads/main/%EB%8D%B0%EC%9D%B4%ED%84%B0/ex.jpeg)

![image](https://raw.githubusercontent.com/simsimee/HKNU_lecture/refs/heads/main/%EB%8D%B0%EC%9D%B4%ED%84%B0/eee.jpeg)

### 여러분들은 아래 사진들을 보고 어느정도의 Confidence로 구분할 수 있나요?
![confidence](https://raw.githubusercontent.com/simsimee/HKNU_lecture/refs/heads/main/%EB%8D%B0%EC%9D%B4%ED%84%B0/_107748428_c9oqh-2w3g-1ayj08mjylwlpi46qabxgtyqa.jpg)

### 🌟 YOLO의 특징
- **빠른 속도**: 실시간 처리 가능 (30+ FPS)
- **높은 정확도**: 최신 딥러닝 기술 적용
- **쉬운 사용**: 사전 훈련된 모델 제공

----
## 🔧 환경 설정 및 라이브러리 설치

### 📦 필요한 라이브러리
- **ultralytics**: YOLO 최신 버전 (YOLOv8)
- **opencv-python**: 이미지/비디오 처리
- **pillow**: 이미지 편집
- **matplotlib**: 결과 시각화
- **torch**: 딥러닝 프레임워크

In [ ]:

# ultralytics 설치 (YOLOv8 포함) – 약 1분
# pip install: 패키지 설치 명령어
# ultralytics: YOLO 모델이 포함된 라이브러리
# 나머지는 이미지 처리와 시각화를 위한 보조 라이브러리들
!pip install ultralytics>=8.0.0 ipywidgets matplotlib seaborn pillow opencv-python koreanize_matplotlib

!apt-get update -y
!apt-get install -y fonts-nanum
!fc-cache -fv


In [ ]:

# 라이브러리 불러오기 (import)
import ultralytics           # YOLO 모델 라이브러리
from ultralytics import YOLO # YOLO 클래스만 따로 불러오기
import torch                 # 딥러닝 프레임워크
import os                    # 파일/폴더 관련 기능
import matplotlib.pyplot as plt  # 그래프 그리기
import seaborn as sns        # 예쁜 그래프 그리기
import pandas as pd          # 데이터 분석 라이브러리

# 설치된 버전 확인 - 잘 설치되었는지 체크
print('Ultralytics version:', ultralytics.__version__)
print('PyTorch version    :', torch.__version__)
print('CUDA available     :', torch.cuda.is_available())  # True면 GPU 사용 가능

# GPU가 있으면 GPU(0번) 사용, 없으면 CPU 사용
device = 0 if torch.cuda.is_available() else 'cpu'
print('사용할 장치:', 'GPU' if device == 0 else 'CPU')

---
# 🖼️ 1. 컴퓨터 비전 기초 실습

## 📚 이미지의 기본 구조
- 이미지는 **픽셀(Pixel)**들의 2차원 배열
- 컬러 이미지는 **RGB(빨강, 초록, 파랑)** 3개의 채널로 구성
- 각 채널의 값은 **0~255** 사이의 숫자

### 🎨 이미지 표현 방식
```
흑백 이미지: (세로, 가로)          예: (100, 200) = 100x200 픽셀
컬러 이미지: (세로, 가로, 채널)     예: (100, 200, 3) = RGB 컬러 이미지
```

In [ ]:

# 필요한 라이브러리 불러오기
import cv2                  # OpenCV - 이미지 처리 라이브러리
import numpy as np          # NumPy - 숫자 배열 처리
import urllib.request   # 인터넷에서 파일 다운로드

# 시각화 라이브러리
import matplotlib.pyplot as plt  # 그래프 그리기
# 진행 상황 표시
from tqdm.notebook import tqdm   # 프로그레스 바 표시
tqdm.pandas(desc="토큰화 진행")   # pandas에 적용

# 한글 폰트 설정
import koreanize_matplotlib      # matplotlib 한글 깨짐 방지
%matplotlib inline

import matplotlib.font_manager as fm
import matplotlib as mpl


# 샘플 이미지 다운로드 (귀여운 강아지 사진)
image_url = "https://raw.githubusercontent.com/simsimee/HKNU_lecture/refs/heads/main/%EB%8D%B0%EC%9D%B4%ED%84%B0/%EC%98%88%EC%8B%9C_%EC%9D%B4%EB%AF%B8%EC%A7%80/%EA%B0%95%EC%95%84%EC%A7%803.jpeg"
urllib.request.urlretrieve(image_url, "sample_dog.jpg")
print("✅ 샘플 이미지 다운로드 완료!")

# 이미지 읽기
# cv2.imread(): 이미지 파일을 읽어서 숫자 배열로 변환
image = cv2.imread("sample_dog.jpg")

# 이미지 정보 확인
print(f"📊 이미지 형태 (shape): {image.shape}")
print(f"   - 세로(높이): {image.shape[0]} 픽셀")
print(f"   - 가로(너비): {image.shape[1]} 픽셀")
print(f"   - 채널 수: {image.shape[2]} (BGR = 파랑, 초록, 빨강)")
print(f"📊 이미지 데이터 타입: {image.dtype}")
print(f"📊 픽셀 값 범위: {image.min()} ~ {image.max()}")

In [ ]:
# 이미지 표시하기
# 왼쪽: BGR (OpenCV 원본) - 색상이 이상하게 보임
# 오른쪽: RGB (변환 후) - 정상적인 색상

# BGR에서 RGB로 변환
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# 2개의 이미지를 나란히 표시
plt.figure(figsize=(12, 5))

# 왼쪽: BGR (원본)
plt.subplot(1, 2, 1)  # 1행 2열 중 1번째
plt.imshow(image)     # BGR 그대로 표시 (색상 이상함)
plt.title("BGR (OpenCV 원본)\n색상이 이상해 보임", fontsize=12)
plt.axis('off')

# 오른쪽: RGB (변환 후)
plt.subplot(1, 2, 2)  # 1행 2열 중 2번째
plt.imshow(image_rgb) # RGB로 변환 후 표시
plt.title("RGB (변환 후)\n정상적인 색상", fontsize=12)
plt.axis('off')

plt.tight_layout()
plt.show()

print("💡 OpenCV는 BGR, Matplotlib은 RGB를 사용하므로 변환이 필요합니다!")

### 🔄 이미지 변환 실습


In [ ]:
# 1. 그레이스케일(흑백) 변환
gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

print(f"📊 원본 이미지 형태: {image.shape} (높이, 너비, 채널)")
print(f"📊 흑백 이미지 형태: {gray_image.shape} (높이, 너비) - 채널이 없음!")

# 흑백 이미지 표시
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(image_rgb)
plt.title("원본 컬러 이미지", fontsize=12)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(gray_image, cmap='gray')  # cmap='gray': 흑백으로 표시
plt.title("그레이스케일 이미지", fontsize=12)
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:

# 2. 이미지 크기 조정 (Resize)
# cv2.resize(이미지, (새로운 너비, 높이))

# 다양한 크기로 변환해보기
small_image = cv2.resize(image_rgb, (200, 150))    # 작게
medium_image = cv2.resize(image_rgb, (400, 300))   # 중간
yolo_size = cv2.resize(image_rgb, (640, 640))      # YOLO 입력 크기 (정사각형)

print("📊 크기 조정 결과:")
print(f"   원본: {image_rgb.shape}")
print(f"   작게: {small_image.shape}")
print(f"   중간: {medium_image.shape}")
print(f"   YOLO용: {yolo_size.shape}")

# 크기별 비교 표시
plt.figure(figsize=(14, 4))

plt.subplot(1, 4, 1)
plt.imshow(image_rgb)
plt.title(f"원본\n{image_rgb.shape[1]}x{image_rgb.shape[0]}", fontsize=10)
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(small_image)
plt.title(f"작게\n200x150", fontsize=10)
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(medium_image)
plt.title(f"중간\n400x300", fontsize=10)
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(yolo_size)
plt.title(f"YOLO용 (정사각형)\n640x640", fontsize=10)
plt.axis('off')

plt.tight_layout()
plt.show()

print("💡 YOLO는 정사각형 이미지(640x640)를 입력으로 사용합니다!")

In [ ]:

# 3. 이미지 자르기 (Cropping)
# 파이썬 슬라이싱: image[y시작:y끝, x시작:x끝]

# 이미지의 중앙 부분만 자르기
h, w = image_rgb.shape[:2]  # 높이, 너비 가져오기

# 중앙 영역 계산 (전체의 50%)
margin_y = h // 4
margin_x = w // 4

# 자르기 실행
cropped_center = image_rgb[margin_y:h-margin_y, margin_x:w-margin_x]

# 상단 절반만 자르기
cropped_top = image_rgb[:h//2, :]

# 왼쪽 절반만 자르기
cropped_left = image_rgb[:, :w//2]

print("📊 자르기 결과:")
print(f"   원본: {image_rgb.shape}")
print(f"   중앙: {cropped_center.shape}")
print(f"   상단: {cropped_top.shape}")
print(f"   왼쪽: {cropped_left.shape}")

# 결과 비교 표시
plt.figure(figsize=(14, 4))

plt.subplot(1, 4, 1)
plt.imshow(image_rgb)
plt.title("원본", fontsize=12)
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(cropped_center)
plt.title("중앙 부분", fontsize=12)
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(cropped_top)
plt.title("상단 절반", fontsize=12)
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(cropped_left)
plt.title("왼쪽 절반", fontsize=12)
plt.axis('off')

plt.tight_layout()
plt.show()

### 🎨 이미지 필터 적용

In [ ]:

# 4. 블러(흐림) 필터 적용
# cv2.GaussianBlur(이미지, (커널크기), 표준편차)
# 커널 크기가 클수록 더 흐려짐 (홀수만 사용 가능: 3, 5, 7, 9...)

blur_light = cv2.GaussianBlur(image_rgb, (5, 5), 0)     # 약한 블러
blur_medium = cv2.GaussianBlur(image_rgb, (15, 15), 0)  # 중간 블러
blur_strong = cv2.GaussianBlur(image_rgb, (31, 31), 0)  # 강한 블러

# 결과 비교 표시
plt.figure(figsize=(14, 4))

plt.subplot(1, 4, 1)
plt.imshow(image_rgb)
plt.title("원본", fontsize=12)
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(blur_light)
plt.title("약한 블러 (5x5)", fontsize=12)
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(blur_medium)
plt.title("중간 블러 (15x15)", fontsize=12)
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(blur_strong)
plt.title("강한 블러 (31x31)", fontsize=12)
plt.axis('off')

plt.tight_layout()
plt.show()

print("💡 블러는 노이즈 제거, 개인정보 보호, 배경 흐림 효과 등에 활용됩니다.")

In [ ]:
# 5. 엣지(경계선) 검출
# cv2.Canny(이미지, 낮은임계값, 높은임계값)
# 임계값: 엣지로 판단하는 기준 (낮을수록 더 많은 엣지 검출)

edges_weak = cv2.Canny(gray_image, 50, 150)    # 많은 엣지 검출
edges_normal = cv2.Canny(gray_image, 100, 200) # 보통
edges_strong = cv2.Canny(gray_image, 150, 250) # 강한 엣지만 검출

# 결과 비교 표시
plt.figure(figsize=(14, 4))

plt.subplot(1, 4, 1)
plt.imshow(gray_image, cmap='gray')
plt.title("흑백 원본", fontsize=12)
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(edges_weak, cmap='gray')
plt.title("엣지 검출 (약한 기준)\n더 많은 선 검출", fontsize=10)
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(edges_normal, cmap='gray')
plt.title("엣지 검출 (보통)", fontsize=12)
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(edges_strong, cmap='gray')
plt.title("엣지 검출 (강한 기준)\n주요 선만 검출", fontsize=10)
plt.axis('off')

plt.tight_layout()
plt.show()

print("💡 엣지 검출은 물체의 윤곽선을 찾아내어 객체 탐지의 기초가 됩니다!")

### ✅ 컴퓨터 비전 기초 실습 정리

| 배운 내용 | 사용한 함수 | 활용 예시 |
|----------|-----------|----------|
| 이미지 읽기 | `cv2.imread()` | 파일에서 이미지 불러오기 |
| 색상 변환 | `cv2.cvtColor()` | BGR↔RGB, 흑백 변환 |
| 크기 조정 | `cv2.resize()` | 딥러닝 입력 크기 맞추기 |
| 이미지 자르기 | 슬라이싱 `[y:y, x:x]` | 관심 영역 추출 |
| 블러 처리 | `cv2.GaussianBlur()` | 노이즈 제거, 개인정보 보호 |
| 엣지 검출 | `cv2.Canny()` | 물체 윤곽선 찾기 |

---

# 🎯 2. YOLO 객체 탐지 실습
## 2. 데이터셋 – COCO128  📊
- **COCO128**은 COCO 2017 train split 중 앞 128장을 뽑은 소규모 튜토리얼용 데이터셋
- 자동으로 다운로드·전처리


In [ ]:
# ⬇️ coco128 데이터셋 다운로드
import urllib.request  # 인터넷에서 파일 다운로드
import zipfile         # ZIP 파일 압축 해제
import os              # 파일/폴더 관리
import pathlib         # 경로 처리
import shutil          # 파일 복사/이동

# 다운로드 URL
url = "https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip"

# 저장할 폴더 생성
root = pathlib.Path('datasets')
root.mkdir(parents=True, exist_ok=True)  # 폴더가 없으면 생성
zip_path = root / 'coco128.zip'

# 다운로드 시작
print("📥 Downloading coco128.zip ...")
urllib.request.urlretrieve(url, zip_path)
print("✅ Download complete! Extracting...")

# ZIP 압축 해제
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(root)

# ZIP 파일 삭제 (공간 절약)
zip_path.unlink()
print("✅ Dataset ready at", root / 'coco128')

In [ ]:
import random  # 무작위 선택
import glob    # 파일 패턴 검색

# 모든 학습 이미지 경로 가져오기
# glob.glob(): 패턴에 맞는 모든 파일 경로를 리스트로 반환
train_imgs = glob.glob('datasets/coco128/images/train2017/*.jpg')
print(f"총 이미지 수: {len(train_imgs)}장")

# 4장을 무작위로 선택
samples = random.sample(train_imgs, 4)

# 선택한 이미지들 표시
plt.figure(figsize=(8, 8))
for i, img_path in enumerate(samples):
    # 이미지 읽고 BGR→RGB 변환
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

    # 2x2 격자에 이미지 배치
    plt.subplot(2, 2, i+1)
    plt.imshow(img)

    # 파일명만 추출해서 제목으로 표시
    filename = img_path.split('/')[-1]
    plt.title(f"{filename}", fontsize=8)
    plt.axis('off')

plt.suptitle("COCO128 샘플 이미지", fontsize=14)
plt.tight_layout()
plt.show()



## 3. 모델 학습 📚
### mAP (mean Average Precision) 🏆
- mAP는 객체 탐지(Object Detection) 모델의 정밀도(Precision)와 재현율(Recall)을 종합적으로 평가하는 대표 지표
- 하나의 클래스에 대해 IoU(Intersection‑over‑Union) 임계값을 고정한 뒤 Precision–Recall 곡선 아래 넓이(Area Under Curve)를 적분하면 AP(Average Precision)
- 모든 클래스에 대해 평균 내면 mAP가 됩니다.

#### AP 계산 절차 🎯

1. 예측 박스(Confidence 내림차순)와 GT 박스 매칭
2. IoU ≥ 임계값(예: 0.5)이면 TP, 그렇지 않으면 FP
3. 누적 TP·FP로 Precision, Recall을 계산
4. PR 곡선을 0 → 1까지 101 점(0, 0.01, … 1.0)에서 샘플링해 적분

### Confidence & IoU 🎯
- Confidence : 모델이 객체를 얼마나 확신하여 예측했는지를 나타내는 정도 (1에 가까울수록 확신)
- IoU : GT(정답)와 예측한 객체의 Box의 겹치는 영역을 전체 합집합으로 나눈 값 (1에 가까울수록 완전 일치)
![IoU](https://raw.githubusercontent.com/simsimee/HKNU_lecture/refs/heads/main/%EB%8D%B0%EC%9D%B4%ED%84%B0/iou.png)

#### Precision & Recall 계산 🎯

- Precision (정밀도) = TP / (TP + FP)
    - 모델이 '검출했다' 고 예측한 것 중 실제로 맞은 비율 ✅

- Recall (재현율) = TP / (TP + FN)
    - GT(Ground Truth, 정답) 중에서 모델이 놓치지 않고 검출한 비율  🔍

![Confusion_Matrix](https://raw.githubusercontent.com/simsimee/HKNU_lecture/refs/heads/main/%EB%8D%B0%EC%9D%B4%ED%84%B0/91576928-fa257300-e982-11ea-9b99-3bde26dd4539.png)

### 값 해석하기 ✅

- 1.0 (100 %)에 가까울수록 검출 성능이 좋음 🌟

In [ ]:
# ‼️ 에포크 수를 늘릴수록 성능은 좋아지지만 시간이 오래 걸립니다.
model = YOLO('yolov8n.pt')        # nano 모델 (≈3.2 M 파라미터)
results = model.train(
    data='coco128.yaml',          # 내장 데이터셋 경로
    epochs=20,                     # 데모용 5 epoch
    imgsz=640,
    device=device
)

## 4. 예측 결과 시각화 📚


In [ ]:

from PIL import Image  # 이미지 표시를 위한 라이브러리

# 테스트할 이미지 URL (버스가 있는 거리 사진)
sample_img = 'https://ultralytics.com/images/bus.jpg'

# 모델로 객체 탐지 수행
# predict(): 이미지에서 물체 찾기
# save=False: 결과를 파일로 저장하지 않음
# imgsz=640: 640x640 크기로 분석
# conf=0.50: 50% 이상 확신하는 것만 표시
pred = model.predict(sample_img, save=False, imgsz=640, conf=0.50)

# 첫 번째 결과 가져오기
res = pred[0]

# 결과 시각화
# plot(): 원본 이미지에 탐지 결과(박스, 라벨)를 그려서 반환
img_bgr = res.plot(boxes=True, masks=True, probs=False, labels=True)
img_rgb = Image.fromarray(img_bgr[..., ::-1])  # BGR→RGB 변환
display(img_rgb)

# 탐지된 객체 정보 출력
print(f"\n📊 탐지된 객체 수: {len(res.boxes)}개")

<div style="margin:20px 0 4px;padding:14px 20px;border-radius:10px;background:#F4E7DF;border:1px solid #E7DCC6;">
<span style="font-size:19px;font-weight:800;color:#23211C;">✏️ 직접 해보기 — 연습문제</span><br>
<span style="color:#5A5448;font-size:14px;line-height:1.7;">이 자료는 코드를 <b>돌려보고 결과를 관찰</b>하는 데모입니다. 아래 연습은 위에서 만든 변수(<code>model</code>, <code>sample_img</code>, <code>pipe</code> 등)를 그대로 쓰며, <b>값 하나만 바꿔</b> 결과가 어떻게 달라지는지 직접 확인하는 것이 목표입니다. 빈칸(<code>____</code>)을 채워 본 뒤 <b>정답 노트북</b>과 비교하세요.</span>
</div>

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 1 Confidence 임계값 바꿔 추론하기 <span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">빈칸 채우기</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 위와 같은 버스 이미지(<code>sample_img</code>)를 <code>conf</code> 값만 <b>0.25</b>로 낮춰 다시 추론하고, 탐지된 객체 수가 어떻게 달라지는지 확인하세요.<br>
<b>힌트</b> &nbsp; <code>model.predict(이미지, imgsz=640, conf=____)</code> · 결과 개수는 <code>len(res.boxes)</code><br>
<b>예상</b> &nbsp; 임계값이 낮아져 <b>더 많은 박스</b>가 잡힌다(확신이 낮은 객체까지 포함)
</div>
</div>

In [ ]:
# ✏️ 연습문제 1 — 빈칸(____)을 채워 보세요
# conf 값만 0.25로 낮춰 같은 이미지를 다시 추론
pred_low = model.predict(sample_img, save=False, imgsz=640, conf=____)
res_low = pred_low[0]
print("conf=0.25 탐지 객체 수:", len(res_low.boxes), "개")

## 5. 학습 곡선 분석 📈


In [ ]:

# 학습 결과 CSV 파일 읽기
logfile = 'runs/detect/train/results.csv'
df = pd.read_csv(logfile)

print("📊 학습 결과 데이터 미리보기:")
print(df[['epoch', 'train/box_loss', 'train/cls_loss', 'metrics/mAP50-95(B)']].head())

# 그래프 그리기
plt.figure(figsize=(10, 5))

# 각 지표별로 선 그래프 그리기
plt.plot(df['epoch'], df['train/box_loss'], marker='o', label='Box Loss (위치 오류)', linewidth=2)
plt.plot(df['epoch'], df['train/cls_loss'], marker='s', label='Class Loss (분류 오류)', linewidth=2)
plt.plot(df['epoch'], df['metrics/mAP50-95(B)'], marker='^', label='mAP50-95 (정확도)', linewidth=2)

# 그래프 꾸미기
plt.xlabel('Epoch (학습 반복 횟수)', fontsize=12)
plt.ylabel('값', fontsize=12)
plt.legend(loc='center right', fontsize=10)
plt.title('📈 학습 곡선 (Training Curves)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 해석:")
print("   - Loss가 감소하면 → 모델이 점점 정확해지고 있음")
print("   - mAP가 증가하면 → 물체 탐지 성능이 향상되고 있음")

## 6. Future work  
1. **에포크·배치 크기 조정** – 더 많은 학습으로 성능 향상 시도  
2. **자체 데이터셋 사용** – `data=` 항목에 여러분의 YAML 설정 파일 지정  
3. **모델 크기 실험** – `yolov8s.pt`, `yolov8m.pt`, `yolov8l.pt` 등 사용해 보기  
4. **하이퍼파라미터 튜닝** – `lr0`, `weight_decay` 등 인자 튜닝으로 모델 성능 향상

---------
## ✅ ++추가++ 이미지 생성 실습
### Text-to-Image 생성 모델 (허깅페이스) 🖼️
[허깅페이스 링크](https://huggingface.co/)

- 허깅페이스란?
  - 다양한 오픈소스 모델들의 가중치를 사용하기 편하게 만들어주는 플랫폼
- Text-to-image 생성 모델이란?
  - 자연어 프롬프트를 입력하면 해당 설명과 유사하도록 이미지를 생성해주는 이미지 생성 AI

### 1. 환경구성


In [ ]:

# diffusers, accelerate, transformers 설치 – 약 1분
!pip install --upgrade diffusers[torch] transformers accelerate safetensors


In [ ]:

# 로그인창이 뜨면 본인의 허깅페이스 토큰(hf_...)을 발급받아 붙여넣으세요
# 토큰 발급: huggingface.co → Settings → Access Tokens
from huggingface_hub import login
login()


### 2. 모델 로드
사용하는 모델 : StableDiffusion

[모델 허깅페이스 링크](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5)

오픈소스 Text-to-image 모델로, 이미지 생성 모델에 있어서는 Yolo처럼 넓은 커뮤니티를 가지고 가장 활발하게 활용되고 있는 모델 중 하나

In [ ]:

import torch
from diffusers import StableDiffusionPipeline

# Stable Diffusion 모델 ID
model_id = "runwayml/stable-diffusion-v1-5"

# 모델 불러오기
# from_pretrained(): 미리 학습된 모델 가져오기
# torch_dtype=torch.float16: 메모리 절약을 위한 반정밀도 사용
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
)

# GPU가 있으면 GPU로, 없으면 CPU로 실행
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = pipe.to(device)

print(f"✅ 모델 로드 완료! 사용 장치: {device}")


----
### 이미지 생성 파라미터

- Prompt / Negative Prompt : 모델의 ‘명령문’. 가장 큰 영향도를 갖습니다.
- Seed : 난수 초기값. 같은 시드는 동일 결과 재현을 가능하게 합니다.
- Guidance Scale (CFG) : 프롬프트를 얼마나 강하게 따를지(보통 5 – 15, SD 기본 7.5 정도).
- Inference Steps : 확산 단계 수. Steps↑ → 세부 묘사 향상, 연산 시간 증가(20 – 50 권장).

튜닝 팁
- 하나씩 바꿔보기(A/B test) → 변경이 품질에 미친 영향 파악이 쉬움
- Seed 고정 + 파라미터만 변경 → 결과 시각적 비교가 명확

In [ ]:

# 시드(seed) 고정 - 같은 결과를 재현하기 위해
seed = 107
generator = torch.Generator(device="cuda").manual_seed(seed)

# 프롬프트: 생성하고 싶은 이미지를 영어로 설명
# 여러분도 다른 문장으로 바꿔보세요!
prompt = "a cute dog, realistic"
print(f"🎨 생성 중... 프롬프트: '{prompt}'")

# 이미지 생성
# num_inference_steps: 생성 단계 수 (높을수록 품질 좋지만 느림)
# guidance_scale: 프롬프트를 얼마나 따를지 (7.5가 기본값)
image = pipe(
    prompt,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=generator
).images[0]

# 결과 보기
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.title(f"🖼️ 생성된 이미지: '{prompt}'", fontsize=12)
plt.axis('off')
plt.show()

print("✅ 이미지 생성 완료!")

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 2 생성 프롬프트 바꿔 보기 <span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">빈칸 채우기</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 위 이미지 생성 코드에서 <code>prompt</code>만 <b>내가 원하는 영어 문장</b>으로 바꿔 새 이미지를 생성하세요(<code>seed</code>는 그대로 두어 차이가 프롬프트 때문임을 확인).<br>
<b>힌트</b> &nbsp; <code>prompt = "____"</code>에 영어로 설명(예: <code>"a cute cat on the sofa, realistic"</code>)<br>
<b>예상</b> &nbsp; 프롬프트 설명에 맞는 새 이미지가 생성된다
</div>
</div>

In [ ]:
# ✏️ 연습문제 2 — 빈칸(____)을 채워 보세요
# seed 는 그대로 두고 prompt 만 바꿔 차이를 비교
generator = torch.Generator(device="cuda").manual_seed(107)
prompt = "____"          # 예: "a cute cat on the sofa, realistic"
my_image = pipe(prompt, num_inference_steps=30, guidance_scale=7.5,
                generator=generator).images[0]

plt.figure(figsize=(8, 8))
plt.imshow(my_image); plt.title(prompt); plt.axis('off'); plt.show()

#### 학습한 모델로 생성된 이미지 추론하기


In [ ]:

from PIL import Image

# 생성된 이미지로 객체 탐지 수행
pred = model.predict(image, save=False, imgsz=640, conf=0.50)
res = pred[0]

# 결과 시각화
img_bgr = res.plot(boxes=True, masks=True, probs=False, labels=True)
img_rgb = Image.fromarray(img_bgr[..., ::-1])
display(img_rgb)

print("💡 AI가 생성한 이미지에서도 YOLO가 객체를 탐지했습니다!")

#### 이미지 생성 모델의 활용

1. 데이터 증강 : 검출하기 어려운 각도·배경·조명 상태 이미지를 합성해 YOLO 학습 데이터에 추가 ➜ 특히 소수 클래스 균형 개선.

2. Rare‑case 시뮬레이션 : 실제 촬영이 힘든 위험·야간·악천후 상황을 가상 생성.

3. 컨셉 아트·스토리보드 : 제품·UI mock‑up, 광고·마케팅 크리에이티브 빠른 프로토타입.

4. 교육·연구 : 모델 베이스라인 구축, Zero‑shot·Few‑shot 실험용 synthetic data.

## 🌄 각자 원하는 이미지 추론해보기
- 각자의 반려견, 반려묘나 주변인, 물건들의 사진을 업로드하여 한 번씩 추론해보며 모델의 성능을 체크해보세요
- 업로드 후 파일에 마우스 오른쪽 클릭하여 경로복사 후 아래 코드에 경로를 입력하고, 위의 코드를 참고하여 추론해보세요

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 3 내 이미지로 추론하기 <span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">직접 작성</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 반려동물·주변 물건 등 <b>내 사진 한 장</b>을 Colab에 업로드하고, <code>model.predict()</code>로 추론해 결과를 시각화하세요.<br>
<b>힌트</b> &nbsp; 왼쪽 폴더 아이콘에 파일 업로드 → 경로 복사 후 <code>model.predict('내경로.jpg', imgsz=640, conf=0.50)</code> · 위 시각화 셀(<code>res.plot</code>)을 그대로 참고<br>
<b>예상</b> &nbsp; 사진 속 객체에 박스·라벨이 그려진다(없는 클래스는 못 잡을 수 있음)
</div>
</div>

In [ ]:
# ✏️ 직접 작성 — 내 이미지를 업로드해 model.predict 로 추론하고 결과를 시각화하세요
# (힌트) my_path = '내가_업로드한_파일.jpg'
#        pred = model.predict(my_path, save=False, imgsz=640, conf=0.50)
#        위의 res.plot(...) 시각화 코드를 참고
